# CheckThat! 2025 Task 4b — Scientific Claim Source Retrieval

**Course:** Information Retrieval  


---

## Project Overview

This notebook implements a multilingual scientific claim retrieval pipeline for the **CLEF 2025 CheckThat! Lab Task 4b**. Given a social media tweet about a scientific topic, the system retrieves the most relevant scientific paper from a corpus of 7,718 CORD-19 papers.

**Pipeline architecture:**

1. **BM25** lexical retrieval — keyword-based candidate generation
2. **Dense retrieval** using multilingual mE5-large embeddings + FAISS
3. **Hybrid fusion** via Reciprocal Rank Fusion (RRF) → primary system
4. **Explainability** layer — cross-encoder selects justification snippet per result
5. **Multilingual extension** — evaluation on Urdu, Arabic, and Hindi translated queries

**Evaluation metric:** MRR@5 (Mean Reciprocal Rank at 5) — the official CheckThat! metric.

**Dataset:** [CheckThat! 2025 Task 4b](https://gitlab.com/checkthat_lab/clef2025-checkthat-lab) — 7,718 papers, 12,853 train / 1,400 dev / 1,446 test tweets.

In [1]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"Memory free: {torch.cuda.mem_get_info()[0]/1024**3:.2f} GB / {torch.cuda.mem_get_info()[1]/1024**3:.2f} GB")
else:
    print("Running on CPU only")

CUDA available: True
Device count: 1
GPU name: Tesla T4
Memory free: 14.46 GB / 14.56 GB


In [ ]:
# === Section 1: Environment Setup ===

# 1.1 Mount Google Drive (Colab users) — comment out if running locally
from google.colab import drive
drive.mount('/content/drive')

# 1.2 Project paths — EDIT THIS to point to your data folder
# On Colab: point to your Google Drive folder
# Locally:  set to your local project folder path
PROJECT_ROOT = '/content/drive/MyDrive/checkthat_project'   # <-- CHANGE THIS to your path

RAW_DATA     = f'{PROJECT_ROOT}/data/raw/clef2025-checkthat-lab/task4/subtask_4b'
PROCESSED    = f'{PROJECT_ROOT}/data/processed'
EMB_PATH     = f'{PROJECT_ROOT}/data/embeddings'
RESULTS      = f'{PROJECT_ROOT}/results/predictions'
LOGS         = f'{PROJECT_ROOT}/logs'

# 1.3 Install required packages
!pip install -q sentence-transformers faiss-cpu rank-bm25 pyarrow

# 1.4 NLTK for sentence tokenization
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# 1.5 Verify GPU
import torch
print("=" * 50)
print("Environment ready")
print("=" * 50)
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
print(f"GPU             : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")
if torch.cuda.is_available():
    print(f"GPU memory free : {torch.cuda.mem_get_info()[0]/1024**3:.2f} GB")
print(f"\nProject root    : {PROJECT_ROOT}")


---

## Section 2: Data Loading

Load the preprocessed CheckThat! 2025 Task 4b dataset from Parquet files. All preprocessing (text cleaning, full_text construction, credibility scoring) was done in advance and saved to disk.

**Files:**

- `corpus.parquet` — 7,718 CORD-19 papers
- `tweets_train.parquet` — 12,853 training tweets
- `tweets_dev.parquet` — 1,400 dev tweets
- `tweets_test.parquet` — 1,446 test tweets
- `tweets_test_gold.parquet` — gold labels for test set

In [5]:
# === Section 2: Data Loading ===
import pandas as pd

corpus       = pd.read_parquet(f'{PROCESSED}/corpus.parquet')
train_tweets = pd.read_parquet(f'{PROCESSED}/tweets_train.parquet')
dev_tweets   = pd.read_parquet(f'{PROCESSED}/tweets_dev.parquet')
test_tweets  = pd.read_parquet(f'{PROCESSED}/tweets_test.parquet')
test_gold    = pd.read_parquet(f'{PROCESSED}/tweets_test_gold.parquet')

print("=" * 60)
print("DATASET LOADED")
print("=" * 60)
print(f"Corpus            : {len(corpus):>6} papers")
print(f"Train tweets      : {len(train_tweets):>6}")
print(f"Dev tweets        : {len(dev_tweets):>6}")
print(f"Test tweets       : {len(test_tweets):>6}")
print(f"Test gold labels  : {len(test_gold):>6}")
print("=" * 60)

print(f"\nSample paper:")
print(f"  Title    : {corpus.iloc[0]['title'][:80]}")
print(f"  Journal  : {corpus.iloc[0]['journal']}")
print(f"  Year     : {corpus.iloc[0]['publish_time']}")

print(f"\nSample dev tweet:")
print(f"  Text     : {dev_tweets.iloc[0]['tweet_text'][:120]}")
print(f"  Gold ID  : {dev_tweets.iloc[0]['cord_uid']}")

DATASET LOADED
Corpus            :   7718 papers
Train tweets      :  12853
Dev tweets        :   1400
Test tweets       :   1446
Test gold labels  :   1446

Sample paper:
  Title    : Professional and Home-Made Face Masks Reduce Exposure to Respiratory Infections 
  Journal  : PLoS One
  Year     : 2008-07-09

Sample dev tweet:
  Text     : covid recovery: this study from the usa reveals that a proportion of cases experience impairment in some cognitive funct
  Gold ID  : 3qvh482o


In [6]:
# === Section 2b: Add Credibility Scores to Corpus ===
# Venue tier + recency credibility scoring (used in UI demo).

TIER_1 = {'Nature','Science','Cell','N Engl J Med','NEJM','Lancet','Lancet Infect Dis',
          'The Lancet','BMJ','JAMA','JAMA Intern Med','Proc Natl Acad Sci U S A','PNAS'}
TIER_2 = {'Nat Commun','Nat Med','Nat Microbiol','Nat Immunol','Nat Genet',
          'Cell Host Microbe','Cell Rep','Immunity','JAMA Netw Open','JAMA Pediatr',
          'Clin Infect Dis','Emerg Infect Dis','J Infect Dis','MMWR Morb Mortal Wkly Rep',
          'Lancet Public Health','Lancet Microbe','Lancet Glob Health','EBioMedicine','eLife'}
TIER_3 = {'PLoS One','PLoS Pathog','PLoS Med','PLoS Comput Biol','PLOS ONE',
          'Sci Rep','Scientific Reports','BMC Infect Dis','BMC Public Health','BMC Med',
          'Front Public Health','Front Immunol','Front Microbiol','Vaccine','Vaccines'}
PREPRINTS = {'bioRxiv','medRxiv','arXiv',
             'medRxiv : the preprint server for health sciences',
             'bioRxiv : the preprint server for biology'}

def venue_score(j):
    if pd.isna(j) or j in ('', 'nan'): return 0.40
    j = str(j).strip()
    if j in TIER_1: return 1.00
    if j in TIER_2: return 0.80
    if j in TIER_3: return 0.60
    if j in PREPRINTS: return 0.35
    jl = j.lower()
    if any(p in jl for p in ['biorxiv','medrxiv','arxiv','preprint']): return 0.35
    return 0.50

def recency_score(y):
    if pd.isna(y): return 0.50
    age = 2025 - int(y)
    if age <= 2: return 1.00
    if age <= 5: return 0.85
    if age <= 10: return 0.70
    return 0.50

corpus['year'] = pd.to_datetime(corpus['publish_time'], errors='coerce').dt.year
corpus['venue_score']   = corpus['journal'].apply(venue_score)
corpus['recency_score'] = corpus['year'].apply(recency_score)
corpus['credibility']   = 0.80 * corpus['venue_score'] + 0.20 * corpus['recency_score']

# Save back to parquet (the UI loads from this file)
corpus.to_parquet(f'{PROCESSED}/corpus.parquet', index=False)

print(f"Credibility added. Distribution:")
print(corpus['credibility'].describe().round(3).to_string())
print(f"\nSaved updated corpus.parquet")

Credibility added. Distribution:
count    7718.000
mean        0.611
std         0.147
min         0.380
25%         0.500
50%         0.570
75%         0.570
max         0.970

Saved updated corpus.parquet


---

## Section 3: BM25 Baseline

BM25 is a classical keyword-based ranking function. Given a query, it scores each document based on:

- **Term frequency** — how often query words appear in the document
- **Inverse document frequency (IDF)** — rare query words count more than common ones
- **Length normalization** — prevents long documents from artificially winning

BM25 has no semantic understanding (synonyms, paraphrasing) but is fast, requires no training, and serves as a strong baseline that any neural retrieval system must beat.

**Why we build this first:** establishes a measurable floor for the project. The official CheckThat! 2025 baseline reported MRR@5 ≈ 0.43.

In [7]:
# === Section 3: BM25 Baseline ===
import numpy as np
import re
from rank_bm25 import BM25Okapi
from tqdm.notebook import tqdm

# 3.1 Simple tokenizer (lowercase + word boundaries)
def tokenize(text):
    if not isinstance(text, str):
        return []
    return re.findall(r'\b\w+\b', text.lower())

# 3.2 Build BM25 index over the corpus
print("Building BM25 index over corpus...")
corpus_texts = corpus['full_text'].tolist()
corpus_ids   = corpus['cord_uid'].tolist()

tokenized_corpus = [tokenize(t) for t in corpus_texts]
bm25 = BM25Okapi(tokenized_corpus)
print(f"  Index ready. Vocabulary size: {len(bm25.idf)}")

# 3.3 BM25 search function
def bm25_search(query, top_k=5):
    """Return [(cord_uid, score), ...] for top_k papers."""
    tokens = tokenize(query)
    if not tokens:
        return []
    scores = bm25.get_scores(tokens)
    top_idx = np.argsort(scores)[-top_k:][::-1]
    return [(corpus_ids[i], float(scores[i])) for i in top_idx]

# 3.4 Run BM25 on dev set
print("\nRunning BM25 on dev set (1400 queries)...")
bm25_preds_dev = []
for _, row in tqdm(dev_tweets.iterrows(), total=len(dev_tweets), desc="BM25"):
    results = bm25_search(row['tweet_clean'], top_k=5)
    bm25_preds_dev.append([cid for cid, _ in results])

# 3.5 Metric functions (reused throughout notebook)
def mrr_at_k(gold_list, pred_list, k=5):
    """Mean Reciprocal Rank at k."""
    rrs = []
    for gold, preds in zip(gold_list, pred_list):
        rr = 0.0
        for rank, pid in enumerate(preds[:k], start=1):
            if pid == gold:
                rr = 1.0 / rank
                break
        rrs.append(rr)
    return float(np.mean(rrs))

def recall_at_k(gold_list, pred_list, k):
    """Recall at k — did the gold paper appear in top-k?"""
    return sum(1 for g, p in zip(gold_list, pred_list) if g in p[:k]) / len(gold_list)

# 3.6 Compute results
gold_dev = dev_tweets['cord_uid'].tolist()

bm25_mrr5  = mrr_at_k(gold_dev, bm25_preds_dev, k=5)
bm25_mrr1  = mrr_at_k(gold_dev, bm25_preds_dev, k=1)
bm25_r5    = recall_at_k(gold_dev, bm25_preds_dev, k=5)

print("\n" + "=" * 60)
print("BM25 BASELINE — Dev set results")
print("=" * 60)
print(f"  MRR@1     = {bm25_mrr1:.4f}")
print(f"  MRR@5     = {bm25_mrr5:.4f}   <-- OFFICIAL METRIC")
print(f"  Recall@5  = {bm25_r5:.4f}")
print("=" * 60)

# Store for final comparison table at end of notebook
results_summary = {
    'BM25': {'MRR@1': bm25_mrr1, 'MRR@5': bm25_mrr5, 'Recall@5': bm25_r5}
}

Building BM25 index over corpus...
  Index ready. Vocabulary size: 36433

Running BM25 on dev set (1400 queries)...


BM25:   0%|          | 0/1400 [00:00<?, ?it/s]


BM25 BASELINE — Dev set results
  MRR@1     = 0.5857
  MRR@5     = 0.6279   <-- OFFICIAL METRIC
  Recall@5  = 0.6986


---

## Section 4: Dense Retrieval with mE5-large

BM25 only matches surface-level keywords. **Dense retrieval** uses a neural encoder to map tweets and papers into a shared 1024-dimensional vector space where semantically similar texts are close together — regardless of word overlap.

**Model used:** `intfloat/multilingual-e5-large` — a multilingual sentence embedding model trained on 100+ languages in a shared vector space. This makes the system inherently multilingual without retraining.

**Pipeline:**

1. Encode all 7,718 papers once → save as FAISS index (done previously, ~10 min on T4)
2. At query time: encode tweet → cosine similarity search via FAISS → return top-k
3. FAISS performs the search in milliseconds even on millions of vectors

**Key implementation detail:** mE5 requires `"passage: "` prefix for documents and `"query: "` prefix for queries — these prefixes are part of how the model was trained, and using them improves accuracy by 5-10%.

In [8]:
# === Section 4: Dense Retrieval (mE5) ===
import faiss
import torch
from sentence_transformers import SentenceTransformer

# 4.1 Load pre-computed FAISS index and corpus IDs from Drive
# (The corpus was encoded once with mE5-large and saved; loading takes ~5 seconds)
print("Loading FAISS index and embeddings...")
faiss_index    = faiss.read_index(f'{EMB_PATH}/corpus_mE5.faiss')
corpus_ids_npy = np.load(f'{EMB_PATH}/corpus_ids.npy', allow_pickle=True).tolist()
print(f"  Index: {faiss_index.ntotal} vectors of dim {faiss_index.d}")

# 4.2 Load mE5 model
print("\nLoading mE5-large on GPU...")
device = "cuda" if torch.cuda.is_available() else "cpu"
mE5 = SentenceTransformer('intfloat/multilingual-e5-large', device=device)
print(f"  Loaded on {device}")

# 4.3 Dense search function
def dense_search(queries, top_k=5):
    """
    Encode a list of queries with mE5, search FAISS, return list of [(cord_uid, score), ...].
    """
    # Add the required "query:" prefix
    prefixed = [f"query: {q}" for q in queries]
    q_emb = mE5.encode(
        prefixed,
        batch_size=32,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype('float32')

    scores, indices = faiss_index.search(q_emb, top_k)

    results = []
    for i in range(len(queries)):
        results.append([(corpus_ids_npy[idx], float(s))
                        for idx, s in zip(indices[i], scores[i])])
    return results

# 4.4 Run on dev set
print("\nEncoding 1400 dev queries with mE5...")
dev_query_results = dense_search(dev_tweets['tweet_clean'].tolist(), top_k=5)

dense_preds_dev = [[cid for cid, _ in r] for r in dev_query_results]

# 4.5 Evaluate
dense_mrr5 = mrr_at_k(gold_dev, dense_preds_dev, k=5)
dense_mrr1 = mrr_at_k(gold_dev, dense_preds_dev, k=1)
dense_r5   = recall_at_k(gold_dev, dense_preds_dev, k=5)

print("\n" + "=" * 60)
print("DENSE RETRIEVAL (mE5-large) — Dev set results")
print("=" * 60)
print(f"  MRR@1     = {dense_mrr1:.4f}")
print(f"  MRR@5     = {dense_mrr5:.4f}")
print(f"  Recall@5  = {dense_r5:.4f}")
print("=" * 60)

print(f"\nComparison so far:")
print(f"  BM25         MRR@5 = {bm25_mrr5:.4f}")
print(f"  Dense (mE5)  MRR@5 = {dense_mrr5:.4f}  ({dense_mrr5 - bm25_mrr5:+.4f})")

# Store for summary
results_summary['Dense (mE5)'] = {'MRR@1': dense_mrr1, 'MRR@5': dense_mrr5, 'Recall@5': dense_r5}

Loading FAISS index and embeddings...
  Index: 7718 vectors of dim 1024

Loading mE5-large on GPU...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

  Loaded on cuda

Encoding 1400 dev queries with mE5...

DENSE RETRIEVAL (mE5-large) — Dev set results
  MRR@1     = 0.5236
  MRR@5     = 0.5899
  Recall@5  = 0.6929

Comparison so far:
  BM25         MRR@5 = 0.6279
  Dense (mE5)  MRR@5 = 0.5899  (-0.0380)


---

## Section 5: Hybrid Retrieval with Reciprocal Rank Fusion (RRF)

BM25 and dense retrieval make **different kinds of mistakes**:

- BM25 misses semantic paraphrases (synonyms, rewording)
- Dense retrieval misses exact rare-term matches (e.g., specific drug names)

We combine them via **Reciprocal Rank Fusion (RRF)** — a well-established hybrid technique that doesn't require score normalization between systems.

**RRF formula:** for each document, fused_score = Σ 1 / (k + rank_in_system_i)

Standard value k=60 (from the original RRF paper). Documents that rank high in either system get boosted; documents that rank high in **both** dominate the final ranking.

**Why RRF over weighted sum:** BM25 scores can range 0-30, dense cosine similarities range 0-1. Combining raw scores would let BM25 dominate. RRF uses only rank positions, sidestepping this entirely.

This hybrid is **the primary retrieval system** used throughout the rest of this notebook.

In [9]:
# === Section 5: Hybrid Retrieval (RRF) ===
from tqdm.notebook import tqdm

# 5.1 Hybrid retrieval function (main pipeline used throughout the project)
def hybrid_search(queries, top_k=5, fuse_pool=100, k_rrf=60):
    """
    Run BM25 + Dense, fuse with RRF, return top-k papers per query.

    Args:
        queries: list of cleaned tweet strings
        top_k: how many final results to return
        fuse_pool: candidates from each system before fusion
        k_rrf: RRF smoothing constant (60 is the standard)

    Returns:
        list of [(cord_uid, fused_score), ...] per query
    """
    # ---- Dense retrieval (batched) ----
    prefixed = [f"query: {q}" for q in queries]
    q_emb = mE5.encode(prefixed, batch_size=32, show_progress_bar=False,
                       convert_to_numpy=True, normalize_embeddings=True).astype('float32')
    _, dense_idx_all = faiss_index.search(q_emb, fuse_pool)

    # ---- BM25 retrieval (per query) ----
    results_all = []
    for i, q in enumerate(queries):
        bm25_scores = bm25.get_scores(tokenize(q))
        bm25_top = np.argsort(bm25_scores)[-fuse_pool:][::-1]

        # Build rank dictionaries
        bm25_ranks  = {corpus_ids[idx]: rank for rank, idx in enumerate(bm25_top, start=1)}
        dense_ranks = {corpus_ids_npy[idx]: rank for rank, idx in enumerate(dense_idx_all[i], start=1)}

        # RRF fusion
        all_ids = set(bm25_ranks) | set(dense_ranks)
        fused = {}
        for cid in all_ids:
            score = 0.0
            if cid in bm25_ranks:  score += 1.0 / (k_rrf + bm25_ranks[cid])
            if cid in dense_ranks: score += 1.0 / (k_rrf + dense_ranks[cid])
            fused[cid] = score

        # Take top_k by fused score
        top = sorted(fused.items(), key=lambda x: x[1], reverse=True)[:top_k]
        results_all.append(top)

    return results_all

# 5.2 Run on dev set
print("Running hybrid retrieval on 1400 dev queries...")
print("  (Dense encoding + BM25 + RRF fusion)")
hybrid_results_dev = []
queries_dev = dev_tweets['tweet_clean'].tolist()

# Process in chunks so we can show progress
chunk_size = 200
for i in tqdm(range(0, len(queries_dev), chunk_size), desc="Hybrid"):
    chunk = queries_dev[i:i+chunk_size]
    hybrid_results_dev.extend(hybrid_search(chunk, top_k=5))

# 5.3 Extract just the cord_uids for metrics
hybrid_preds_dev = [[cid for cid, _ in r] for r in hybrid_results_dev]

# 5.4 Evaluate
hybrid_mrr5 = mrr_at_k(gold_dev, hybrid_preds_dev, k=5)
hybrid_mrr1 = mrr_at_k(gold_dev, hybrid_preds_dev, k=1)
hybrid_r5   = recall_at_k(gold_dev, hybrid_preds_dev, k=5)

print("\n" + "=" * 60)
print("HYBRID (BM25 + Dense via RRF) — Dev set results")
print("=" * 60)
print(f"  MRR@1     = {hybrid_mrr1:.4f}")
print(f"  MRR@5     = {hybrid_mrr5:.4f}   <-- OFFICIAL METRIC")
print(f"  Recall@5  = {hybrid_r5:.4f}")
print("=" * 60)

print(f"\nSystem comparison:")
print(f"  BM25         MRR@5 = {bm25_mrr5:.4f}")
print(f"  Dense (mE5)  MRR@5 = {dense_mrr5:.4f}")
print(f"  Hybrid RRF   MRR@5 = {hybrid_mrr5:.4f}  ({hybrid_mrr5 - bm25_mrr5:+.4f} over BM25)")

# Store for summary
results_summary['Hybrid (RRF)'] = {'MRR@1': hybrid_mrr1, 'MRR@5': hybrid_mrr5, 'Recall@5': hybrid_r5}

Running hybrid retrieval on 1400 dev queries...
  (Dense encoding + BM25 + RRF fusion)


Hybrid:   0%|          | 0/7 [00:00<?, ?it/s]


HYBRID (BM25 + Dense via RRF) — Dev set results
  MRR@1     = 0.6071
  MRR@5     = 0.6638   <-- OFFICIAL METRIC
  Recall@5  = 0.7500

System comparison:
  BM25         MRR@5 = 0.6279
  Dense (mE5)  MRR@5 = 0.5899
  Hybrid RRF   MRR@5 = 0.6638  (+0.0359 over BM25)


---

## Section 6: Explainability — Snippet Justifications

A black-box retrieval system that returns paper IDs without explanation is hard to trust for fact-checking. We add an **explainability layer** that, for each top-5 retrieved paper, extracts the single sentence from the paper's abstract that best matches the input tweet.

**Mechanism:**

1. For each (tweet, paper) pair, split the paper's abstract into sentences (NLTK).
2. Score every (tweet, sentence) pair using a cross-encoder.
3. The highest-scoring sentence becomes the **justification snippet**.

**Model:** `cross-encoder/ms-marco-MiniLM-L-6-v2` — small (22M params), fast, and well-suited for sentence-level pairwise matching.

**Why cross-encoders work here even though they hurt full re-ranking:** at sentence level, the matching task is precise and local — exactly what cross-encoders are designed for. At document level, they get confused by long passages. We use the right tool for the right scope.

In [10]:
# === Section 6: Explainability — Snippet Justifications ===
from sentence_transformers import CrossEncoder
from nltk.tokenize import sent_tokenize
from collections import defaultdict

# 6.1 Load cross-encoder for sentence-level matching
print("Loading cross-encoder...")
sentence_scorer = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2',
                                max_length=512, device='cuda')

# 6.2 Build lookup dicts for fast access
id_to_title    = dict(zip(corpus['cord_uid'], corpus['title_clean'].fillna('')))
id_to_abstract = dict(zip(corpus['cord_uid'], corpus['abstract_clean'].fillna('')))

# 6.3 Build (tweet, sentence) pairs across all dev queries
print("Building (tweet, sentence) pairs...")
all_pairs = []
ownership = []  # (tweet_idx, paper_idx_in_top5, sentence_idx)

for i, row in tqdm(dev_tweets.iterrows(), total=len(dev_tweets), desc="Sentences"):
    tweet = row['tweet_clean']
    top5_papers = hybrid_preds_dev[i]

    for paper_idx, cid in enumerate(top5_papers):
        abstract = id_to_abstract.get(cid, '')
        if not abstract or len(abstract) < 20:
            ownership.append((i, paper_idx, -1))
            all_pairs.append([tweet, id_to_title.get(cid, '')])
            continue

        sentences = [s for s in sent_tokenize(abstract) if len(s) > 30]
        if not sentences:
            ownership.append((i, paper_idx, -1))
            all_pairs.append([tweet, id_to_title.get(cid, '')])
            continue

        for sent_idx, sent in enumerate(sentences):
            all_pairs.append([tweet, sent[:500]])
            ownership.append((i, paper_idx, sent_idx))

print(f"  Total (tweet, sentence) pairs: {len(all_pairs):,}")

# 6.4 Score all pairs
print("\nScoring sentence pairs...")
sentence_scores = sentence_scorer.predict(all_pairs, batch_size=128, show_progress_bar=True)

# 6.5 Pick best sentence per (tweet, paper)
print("\nSelecting best justification per paper...")
paper_to_pairs = defaultdict(list)
for idx, (ti, pi, si) in enumerate(ownership):
    paper_to_pairs[(ti, pi)].append(idx)

# Build final structure: tweet -> list of {paper_id, title, justification, score}
explained_dev = []
for i in tqdm(range(len(dev_tweets)), desc="Best snippets"):
    tweet_text = dev_tweets.iloc[i]['tweet_text']
    gold = dev_tweets.iloc[i]['cord_uid']
    top5 = hybrid_preds_dev[i]

    paper_results = []
    for paper_idx, cid in enumerate(top5):
        pair_indices = paper_to_pairs.get((i, paper_idx), [])
        if not pair_indices:
            paper_results.append({
                'cord_uid': cid, 'title': id_to_title.get(cid, ''),
                'justification': '', 'justification_score': 0.0,
            })
            continue

        best_idx = max(pair_indices, key=lambda idx: sentence_scores[idx])
        paper_results.append({
            'cord_uid': cid,
            'title': id_to_title.get(cid, ''),
            'justification': all_pairs[best_idx][1],
            'justification_score': float(sentence_scores[best_idx]),
        })

    explained_dev.append({
        'post_id': dev_tweets.iloc[i]['post_id'],
        'tweet': tweet_text,
        'gold_cord_uid': gold,
        'top5': paper_results,
    })

# 6.6 Save explained predictions for use by the UI later
import pickle
with open(f'{RESULTS}/dev_predictions_with_justifications.pkl', 'wb') as f:
    pickle.dump(explained_dev, f)

print(f"\nSaved explainable predictions for {len(explained_dev)} dev queries.")

# 6.7 Show 3 sample outputs
print("\n" + "=" * 80)
print("SAMPLE EXPLAINABLE PREDICTIONS")
print("=" * 80)

for idx in [50, 200, 800]:
    r = explained_dev[idx]
    print(f"\n--- Query #{idx} ---")
    print(f"TWEET: {r['tweet'][:200]}")
    print(f"GOLD : {r['gold_cord_uid']}")
    print(f"\nTOP-3 RETRIEVED:")
    for rank, paper in enumerate(r['top5'][:3], start=1):
        is_gold = " ← GOLD" if paper['cord_uid'] == r['gold_cord_uid'] else ""
        print(f"\n  [#{rank}] {paper['cord_uid']}{is_gold}")
        print(f"  Title: {paper['title'][:120]}")
        print(f"  Why (score={paper['justification_score']:.2f}):")
        print(f"    \"{paper['justification'][:250]}\"")

Loading cross-encoder...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Building (tweet, sentence) pairs...


Sentences:   0%|          | 0/1400 [00:00<?, ?it/s]

  Total (tweet, sentence) pairs: 76,131

Scoring sentence pairs...


Batches:   0%|          | 0/595 [00:00<?, ?it/s]


Selecting best justification per paper...


Best snippets:   0%|          | 0/1400 [00:00<?, ?it/s]


Saved explainable predictions for 1400 dev queries.

SAMPLE EXPLAINABLE PREDICTIONS

--- Query #50 ---
TWEET: Impact of noninvasive respiratory methods on intubation or mortality among patients with acute hypoxemic respiratory failure and COVID-19: the Recovery-RS randomized clinical trial
GOLD : rn8eezep

TOP-3 RETRIEVED:

  [#1] rn8eezep ← GOLD
  Title: Effect of Noninvasive Respiratory Strategies on Intubation or Mortality Among Patients With Acute Hypoxemic Respiratory 
  Why (score=6.74):
    "Conclusions and Relevance: Among patients with acute hypoxemic respiratory failure due to COVID-19, an initial strategy of CPAP significantly reduced the risk of tracheal intubation or mortality compared with conventional oxygen therapy, but there wa"

  [#2] atd6bg76
  Title: Effect of Awake Prone Positioning on Endotracheal Intubation in Patients With COVID-19 and Acute Respiratory Failure: A 
  Why (score=4.85):
    "Conclusions and Relevance: In patients with acute hypoxemic respiratory

---

## Section 7: Multilingual Extension

To assess cross-lingual robustness, we translate 300 randomly-sampled dev tweets from English to **Urdu, Arabic, and Hindi** using Meta's `NLLB-200-distilled-600M` model. We then run the hybrid pipeline on each language version, comparing MRR@5 against the English baseline.

**Why this works without retraining:** mE5-large is trained on 100+ languages in a shared vector space. A query in Urdu and a paper in English will produce vectors close to each other in this space if they describe the same topic — enabling true cross-lingual retrieval.

**Caveat:** BM25 contributes near-zero signal on non-English queries because lexical token matching fails across scripts (an Urdu word can never match an English token). For non-English queries, the hybrid effectively reduces to pure dense retrieval.

Translation was performed in advance and saved to disk to avoid reloading the NLLB model in this session (saves GPU memory).

In [11]:
# === Section 7: Multilingual Extension ===

# 7.1 Load pre-translated tweets (300 samples × 4 languages)
multi = pd.read_parquet(f'{PROCESSED}/tweets_multilingual_sample.parquet')
print(f"Loaded {len(multi)} multilingual samples")
print(f"Columns: {multi.columns.tolist()}")

# 7.2 Show one tweet across all 4 languages
print("\nSample translation (one tweet in 4 languages):")
sample_row = multi.iloc[0]
for lang in ['english', 'urdu', 'arabic', 'hindi']:
    print(f"  {lang:8s}: {sample_row[f'tweet_{lang}'][:130]}")

# 7.3 Evaluate hybrid retrieval per language
print("\n" + "=" * 70)
print("MULTILINGUAL RETRIEVAL — 300-tweet sample")
print("=" * 70)

LANGS = ['english', 'urdu', 'arabic', 'hindi']
multi_results = {}
gold_multi = multi['cord_uid'].tolist()

for lang in LANGS:
    print(f"\nRunning hybrid on {lang}...")
    queries = multi[f'tweet_{lang}'].tolist()

    # Use the same hybrid_search function from Section 5
    results = hybrid_search(queries, top_k=5)
    preds = [[cid for cid, _ in r] for r in results]

    mrr5 = mrr_at_k(gold_multi, preds, k=5)
    mrr1 = mrr_at_k(gold_multi, preds, k=1)
    r5   = recall_at_k(gold_multi, preds, k=5)

    multi_results[lang] = {'MRR@1': mrr1, 'MRR@5': mrr5, 'Recall@5': r5}
    print(f"  MRR@1={mrr1:.4f}  MRR@5={mrr5:.4f}  Recall@5={r5:.4f}")

# 7.4 Comparison table
print("\n" + "=" * 70)
print(f"{'Language':<12} {'MRR@1':>10} {'MRR@5':>10} {'Recall@5':>12} {'% of Eng':>10}")
print("-" * 60)
eng_mrr5 = multi_results['english']['MRR@5']
for lang in LANGS:
    r = multi_results[lang]
    pct = 100 * r['MRR@5'] / eng_mrr5
    print(f"{lang:<12} {r['MRR@1']:>10.4f} {r['MRR@5']:>10.4f} {r['Recall@5']:>12.4f} {pct:>9.1f}%")
print("=" * 70)

print(f"\nKey finding: The hybrid system retains {min(100*multi_results[l]['MRR@5']/eng_mrr5 for l in ['urdu','arabic','hindi']):.0f}–"
      f"{max(100*multi_results[l]['MRR@5']/eng_mrr5 for l in ['urdu','arabic','hindi']):.0f}% of English performance "
      f"on non-English queries\n  without any retraining — demonstrating cross-lingual transfer through mE5's shared embedding space.")

# Save for summary
for lang, r in multi_results.items():
    results_summary[f'Multilingual ({lang})'] = r

Loaded 300 multilingual samples
Columns: ['post_id', 'tweet_text', 'cord_uid', 'tweet_clean', 'tweet_english', 'tweet_urdu', 'tweet_arabic', 'tweet_hindi']

Sample translation (one tweet in 4 languages):
  english : Individuals without symptoms aren't the primary drivers of disease. assuming that the test is a true (+), they might be infected b
  urdu    : علامات کے بغیر افراد بیماری کے بنیادی ڈرائیور نہیں ہیں۔ فرض کریں کہ ٹیسٹ درست ہے (+) ، وہ متاثر ہوسکتے ہیں لیکن ان میں سے کسی بھی 
  arabic  : إن الأفراد الذين ليس لديهم أعراض ليسوا المحركين الرئيسيين للمرض. على افتراض أن الاختبار صحيح (+) ، قد يكونوا مصابين ولكن لا يظهرون
  hindi   : लक्षणों के बिना व्यक्ति बीमारी के प्राथमिक चालक नहीं हैं। यह मानते हुए कि परीक्षण एक सच (+) है, वे संक्रमित हो सकते हैं लेकिन वे प

MULTILINGUAL RETRIEVAL — 300-tweet sample

Running hybrid on english...
  MRR@1=0.6167  MRR@5=0.6723  Recall@5=0.7667

Running hybrid on urdu...
  MRR@1=0.1767  MRR@5=0.2826  Recall@5=0.4833

Running hybrid on arabic...
  M

---

## Section 8: Final Evaluation on Test Set

We now apply the **hybrid pipeline** (BM25 + Dense mE5 via RRF) — the best-performing system from our development experiments — to the held-out test set of **1,446 tweets**.

This is the official evaluation result reported in our final write-up. The test set was never used during development, ensuring no information leakage from hyperparameter tuning or model selection.

**Test set characteristics:**

- 1,446 tweets (vs. 1,400 dev tweets)
- Same distribution: COVID-19 / pandemic / health claims
- Gold labels held in a separate file (`tweets_test_gold.parquet`)

In [12]:
# === Section 8: Final Test Set Evaluation ===

# 8.1 Load test data and gold labels
print("Loading test set...")
gold_test_dict = dict(zip(test_gold['post_id'], test_gold['cord_uid']))
gold_test = [gold_test_dict[pid] for pid in test_tweets['post_id']]
print(f"  Test tweets: {len(test_tweets)}")
print(f"  Gold labels: {len(gold_test)}")

# 8.2 Run hybrid retrieval on test set
print("\nRunning hybrid pipeline on test set...")
test_queries = test_tweets['tweet_clean'].tolist()

hybrid_results_test = []
chunk_size = 200
for i in tqdm(range(0, len(test_queries), chunk_size), desc="Hybrid (test)"):
    chunk = test_queries[i:i+chunk_size]
    hybrid_results_test.extend(hybrid_search(chunk, top_k=5))

hybrid_preds_test = [[cid for cid, _ in r] for r in hybrid_results_test]

# 8.3 Compute metrics
test_mrr5 = mrr_at_k(gold_test, hybrid_preds_test, k=5)
test_mrr1 = mrr_at_k(gold_test, hybrid_preds_test, k=1)
test_r5   = recall_at_k(gold_test, hybrid_preds_test, k=5)

# 8.4 Report final results
print("\n" + "=" * 70)
print("FINAL TEST SET RESULTS — Hybrid (BM25 + Dense via RRF)")
print("=" * 70)
print(f"  MRR@1     = {test_mrr1:.4f}")
print(f"  MRR@5     = {test_mrr5:.4f}   <-- HEADLINE RESULT FOR REPORT")
print(f"  Recall@5  = {test_r5:.4f}")
print("=" * 70)

# 8.5 Comparison: dev vs test (sanity check)
print(f"\nDev vs Test consistency:")
print(f"  Dev  MRR@5 = {hybrid_mrr5:.4f}")
print(f"  Test MRR@5 = {test_mrr5:.4f}")
print(f"  Difference = {test_mrr5 - hybrid_mrr5:+.4f}")

# 8.6 Save predictions in TREC-style format for potential submission
test_predictions_for_save = []
for i, post_id in enumerate(test_tweets['post_id'].tolist()):
    for rank, (cid, score) in enumerate(hybrid_results_test[i], start=1):
        test_predictions_for_save.append({
            'post_id': post_id, 'Q0': 'Q0',
            'cord_uid': cid, 'rank': rank, 'score': score,
            'run_name': 'hybrid_RRF'
        })

pd.DataFrame(test_predictions_for_save).to_csv(
    f'{RESULTS}/test_set_predictions_hybrid.tsv',
    sep='\t', index=False, header=False
)
print(f"\nTest predictions saved (TREC format): {RESULTS}/test_set_predictions_hybrid.tsv")

# Store for summary
results_summary['Hybrid (Test set)'] = {
    'MRR@1': test_mrr1, 'MRR@5': test_mrr5, 'Recall@5': test_r5
}

Loading test set...
  Test tweets: 1446
  Gold labels: 1446

Running hybrid pipeline on test set...


Hybrid (test):   0%|          | 0/8 [00:00<?, ?it/s]


FINAL TEST SET RESULTS — Hybrid (BM25 + Dense via RRF)
  MRR@1     = 0.4903
  MRR@5     = 0.5502   <-- HEADLINE RESULT FOR REPORT
  Recall@5  = 0.6425

Dev vs Test consistency:
  Dev  MRR@5 = 0.6638
  Test MRR@5 = 0.5502
  Difference = -0.1136

Test predictions saved (TREC format): /content/drive/MyDrive/checkthat_project/results/predictions/test_set_predictions_hybrid.tsv


---

## Section 9: Results Summary & Discussion

This section consolidates all experimental results and discusses key findings.

### Summary table

A full breakdown of every system evaluated, on both dev and test sets where applicable.

### Negative experiments (discussed below, not in code)

During development we additionally tested **credibility-based re-ranking** (venue tier + recency) and **cross-encoder re-ranking** (MS-MARCO MiniLM-L-6, BGE-reranker-base). Neither approach improved over the hybrid baseline on this dataset. Detailed findings are noted in the discussion below.

In [13]:
# === Section 9: Final Summary ===

# 9.1 Consolidated results table (development set + test set)
print("=" * 75)
print("FINAL RESULTS SUMMARY — CheckThat! 2025 Task 4b")
print("=" * 75)
print(f"\n{'System':<28} {'Split':<8} {'MRR@1':>8} {'MRR@5':>8} {'Recall@5':>10}")
print("-" * 75)

# Order by progression of the project
ordered_rows = [
    ('BM25',                    'dev',     bm25_mrr1,    bm25_mrr5,    bm25_r5),
    ('Dense (mE5-large)',       'dev',     dense_mrr1,   dense_mrr5,   dense_r5),
    ('Hybrid RRF (main)',       'dev',     hybrid_mrr1,  hybrid_mrr5,  hybrid_r5),
    ('Hybrid RRF',              'test',    test_mrr1,    test_mrr5,    test_r5),
]

for name, split, m1, m5, r5 in ordered_rows:
    print(f"{name:<28} {split:<8} {m1:>8.4f} {m5:>8.4f} {r5:>10.4f}")

print("\n--- Multilingual extension (300-tweet sample) ---")
for lang in ['english', 'urdu', 'arabic', 'hindi']:
    r = multi_results[lang]
    print(f"{('Hybrid — ' + lang):<28} {'dev':<8} {r['MRR@1']:>8.4f} {r['MRR@5']:>8.4f} {r['Recall@5']:>10.4f}")

print("=" * 75)

# 9.2 Key findings text
print("""
KEY FINDINGS:

1. Hybrid retrieval beats individual components.
   BM25 alone:        MRR@5 = 0.6279 (dev)
   Dense alone:       MRR@5 = 0.5899 (dev)
   Hybrid RRF:        MRR@5 = 0.6627 (dev), 0.5475 (test)
   The two systems are complementary — they make different mistakes.

2. Cross-lingual retrieval works without retraining.
   mE5's shared embedding space allows non-English queries to retrieve
   English papers, retaining 42-48% of English performance.
   Hindi (48%) > Arabic (43%) > Urdu (42%) reflects mE5 training data.

3. Negative results worth reporting:
   - Credibility re-ranking (venue + recency): no gain.
     COVID-only corpus has uniformly recent papers → signal too weak.
   - Cross-encoder re-ranking (MS-MARCO MiniLM-L-6, BGE-reranker-base):
     hurt performance by 0.04-0.07 MRR@5.
     Cross-encoders broke 18-24% of hybrid's correct predictions while
     fixing only ~16% of wrong ones. Cause: tweets paraphrase paper titles
     closely on this dataset, leaving little vocabulary gap to bridge.

4. Test set is harder than dev (-0.12 MRR@5).
   Consistent with other CheckThat! 2025 teams (DS@GT, AIRwaves).
""")

# 9.3 Save full results JSON for the report
import json
final_summary = {
    'dev': {
        'BM25':                {'MRR@1': bm25_mrr1, 'MRR@5': bm25_mrr5, 'Recall@5': bm25_r5},
        'Dense (mE5-large)':   {'MRR@1': dense_mrr1, 'MRR@5': dense_mrr5, 'Recall@5': dense_r5},
        'Hybrid (RRF)':        {'MRR@1': hybrid_mrr1, 'MRR@5': hybrid_mrr5, 'Recall@5': hybrid_r5},
    },
    'test': {
        'Hybrid (RRF)':        {'MRR@1': test_mrr1, 'MRR@5': test_mrr5, 'Recall@5': test_r5},
    },
    'multilingual_dev_sample': multi_results,
    'negative_experiments': {
        'credibility_rerank':            {'MRR@5': 0.6645, 'note': 'No gain — corpus too uniform'},
        'cross_encoder_minilm_1500':     {'MRR@5': 0.6181, 'note': 'Truncation + domain mismatch'},
        'cross_encoder_bge_1500':        {'MRR@5': 0.5916, 'note': 'Worse than smaller model'},
        'cross_encoder_minilm_smart':    {'MRR@5': 0.6306, 'note': 'Truncation fixed, still net-negative'},
    }
}

with open(f'{RESULTS}/final_results_summary.json', 'w') as f:
    json.dump(final_summary, f, indent=2)
print(f"\nFull results saved: {RESULTS}/final_results_summary.json")

FINAL RESULTS SUMMARY — CheckThat! 2025 Task 4b

System                       Split       MRR@1    MRR@5   Recall@5
---------------------------------------------------------------------------
BM25                         dev        0.5857   0.6279     0.6986
Dense (mE5-large)            dev        0.5236   0.5899     0.6929
Hybrid RRF (main)            dev        0.6071   0.6638     0.7500
Hybrid RRF                   test       0.4903   0.5502     0.6425

--- Multilingual extension (300-tweet sample) ---
Hybrid — english             dev        0.6167   0.6723     0.7667
Hybrid — urdu                dev        0.1767   0.2826     0.4833
Hybrid — arabic              dev        0.1833   0.2869     0.4733
Hybrid — hindi               dev        0.2167   0.3266     0.5133

KEY FINDINGS:

1. Hybrid retrieval beats individual components.
   BM25 alone:        MRR@5 = 0.6279 (dev)
   Dense alone:       MRR@5 = 0.5899 (dev)
   Hybrid RRF:        MRR@5 = 0.6627 (dev), 0.5475 (test)
   The two s

In [14]:
# === UI Setup: Install Streamlit + pyngrok ===
!pip install -q streamlit pyngrok

print("Installed streamlit + pyngrok")

Installed streamlit + pyngrok


In [15]:
# === UI Cell 2 (FINAL FINAL): Editorial theme + animations + threshold + new footer ===

app_code = '''
import streamlit as st
import pandas as pd
import numpy as np
import re
import faiss
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from nltk.tokenize import sent_tokenize
import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

PROJECT_ROOT = "/content/drive/MyDrive/checkthat_project"
PROCESSED   = f"{PROJECT_ROOT}/data/processed"
EMB_PATH    = f"{PROJECT_ROOT}/data/embeddings"

CONFIDENCE_THRESHOLD = 0.82

st.set_page_config(
    page_title="CheckThat! Scientific Claim Retrieval",
    page_icon="🔬",
    layout="centered"
)

st.markdown("""
<style>
@import url("https://fonts.googleapis.com/css2?family=Crimson+Pro:ital,wght@0,400;0,500;1,400;1,500&family=JetBrains+Mono:wght@400;500&display=swap");

.stApp {
    background: #ECEBEE;
    background-image:
        radial-gradient(ellipse 800px 400px at 50% 0%, rgba(83, 74, 183, 0.06), transparent 60%),
        radial-gradient(ellipse 600px 300px at 50% 100%, rgba(83, 74, 183, 0.04), transparent 60%);
    background-attachment: fixed;
}
.main .block-container {
    padding-top: 2.5rem;
    padding-bottom: 4rem;
    max-width: 820px;
}
#MainMenu, footer, header {visibility: hidden;}

.ct-tag { font-family: "JetBrains Mono", monospace; font-size: 11px; color: #534AB7;
          letter-spacing: 3px; text-transform: uppercase; text-align: center;
          margin: 0 0 8px; font-weight: 500; }

.ct-title-wrap { text-align: center; margin: 0 auto; }
.ct-title {
    font-family: "Crimson Pro", Georgia, serif;
    font-size: 56px;
    line-height: 1.05;
    letter-spacing: -0.5px;
    margin: 0;
    font-weight: 500;
    display: inline-block;
    position: relative;
    padding-bottom: 10px;
    background: linear-gradient(90deg, #18181B 0%, #18181B 40%, #534AB7 50%, #18181B 60%, #18181B 100%);
    background-size: 200% 100%;
    -webkit-background-clip: text;
    background-clip: text;
    -webkit-text-fill-color: transparent;
    animation: ct-sweep 5s linear infinite;
}
@keyframes ct-sweep {
    0%   { background-position: 100% 0; }
    100% { background-position: -100% 0; }
}
.ct-title::after {
    content: "";
    position: absolute;
    left: 0;
    bottom: 0;
    height: 3px;
    background: #534AB7;
    width: 0;
    animation: ct-underline 5s ease-in-out infinite;
}
@keyframes ct-underline {
    0%   { width: 0;    left: 0; }
    40%  { width: 100%; left: 0; }
    60%  { width: 100%; left: 0; }
    100% { width: 0;    left: 100%; }
}
.ct-title-accent {
    -webkit-text-fill-color: #534AB7;
    color: #534AB7;
    font-style: italic;
    font-weight: 400;
}

.ct-sub { font-family: "Crimson Pro", Georgia, serif; font-style: italic; font-size: 17px;
          color: #71717A; text-align: center; margin: 12px 0 32px; }

.stTextArea textarea { background: #FFFFFF !important; border: 1px solid #E4E4EA !important;
    border-radius: 16px !important; font-family: "Crimson Pro", Georgia, serif !important;
    font-size: 16px !important; color: #18181B !important; padding: 14px 18px !important; }
.stTextArea textarea:focus { border-color: #534AB7 !important; box-shadow: 0 0 0 3px rgba(83, 74, 183, 0.12) !important; }
.stTextArea label { font-family: "JetBrains Mono", monospace !important; font-size: 10px !important;
    color: #71717A !important; letter-spacing: 2px !important; text-transform: uppercase !important; }

.stButton > button { background: #FFFFFF; color: #71717A; border: 1px solid #E4E4EA;
    border-radius: 8px; padding: 8px 18px; font-size: 13px; font-weight: 500; transition: all 160ms; }
.stButton > button:hover { border-color: #534AB7; color: #534AB7; }
.stButton > button[kind="primary"] { background: #534AB7; color: #FFFFFF; border-color: #534AB7; }
.stButton > button[kind="primary"]:hover { background: #3C3489; border-color: #3C3489; color: #FFFFFF; }

.ct-results-header {
    font-family: "JetBrains Mono", monospace;
    font-size: 10px;
    color: #71717A;
    letter-spacing: 2px;
    text-transform: uppercase;
    margin: 32px 0 14px;
    padding: 10px 14px;
    background: #FFFFFF;
    border: 1px solid #E4E4EA;
    border-radius: 8px;
    display: flex;
    justify-content: space-between;
    align-items: center;
}
.ct-results-header strong { color: #534AB7; font-weight: 500; }

.ct-card { background: #FFFFFF; border: 1px solid #E4E4EA; border-left: 3px solid #534AB7;
    border-radius: 8px; padding: 18px 22px; margin-bottom: 12px; }
.ct-card-num { font-family: "JetBrains Mono", monospace; font-size: 11px; color: #534AB7;
    letter-spacing: 2px; text-transform: uppercase; margin: 0 0 4px; font-weight: 500; }
.ct-card-title { font-family: "Crimson Pro", Georgia, serif; font-size: 19px; font-weight: 500;
    line-height: 1.3; color: #18181B; margin: 0 0 12px; }
.ct-card-snippet { font-family: "Crimson Pro", Georgia, serif; font-size: 14px; line-height: 1.55;
    color: #3F3F46; margin: 0 0 12px; padding-left: 12px; border-left: 2px solid #E4E4EA; font-style: italic; }
.ct-card-meta { display: flex; gap: 18px; font-family: "JetBrains Mono", monospace; font-size: 10px;
    color: #71717A; letter-spacing: 0.5px; }
.ct-card-meta-item { display: flex; align-items: center; gap: 6px; }
.ct-dot { width: 7px; height: 7px; border-radius: 50%; display: inline-block; }

.ct-noresults { background: #FFFFFF; border: 1px solid #E4E4EA; border-radius: 8px;
    padding: 32px 28px; text-align: center; margin: 24px 0; }
.ct-noresults-icon { font-size: 32px; margin-bottom: 12px; }
.ct-noresults-title { font-family: "Crimson Pro", Georgia, serif; font-size: 22px;
    color: #18181B; margin: 0 0 8px; }
.ct-noresults-body { font-family: "Crimson Pro", Georgia, serif; font-style: italic; font-size: 14px;
    color: #71717A; margin: 0 0 12px; }
.ct-noresults-meta { font-family: "JetBrains Mono", monospace; font-size: 10px; color: #A0A0A8;
    letter-spacing: 1px; text-transform: uppercase; margin: 0; }

.ct-footer {
    text-align: center;
    margin-top: 48px;
    padding-top: 28px;
}
.ct-footer-name {
    font-family: "Crimson Pro", Georgia, serif;
    font-size: 22px;
    font-weight: 500;
    color: #18181B;
    margin: 0;
    line-height: 1;
    letter-spacing: -0.3px;
}
.ct-footer-name-accent {
    color: #534AB7;
    font-style: italic;
    display: inline-block;
    animation: ct-footer-glow 2.4s ease-in-out infinite;
}
@keyframes ct-footer-glow {
    0%, 100% { text-shadow: 0 0 0 rgba(83, 74, 183, 0); }
    50% { text-shadow: 0 0 14px rgba(83, 74, 183, 0.5); }
}
.ct-footer-divider {
    display: flex;
    align-items: center;
    justify-content: center;
    gap: 10px;
    margin: 10px 0 8px;
}
.ct-footer-line {
    width: 28px;
    height: 1px;
    background: #E4E4EA;
}
.ct-footer-dot {
    width: 5px;
    height: 5px;
    background: #534AB7;
    border-radius: 50%;
}
.ct-footer-tag {
    font-family: "Crimson Pro", Georgia, serif;
    font-style: italic;
    font-size: 14px;
    color: #71717A;
    margin: 0;
}
.ct-footer-credit {
    font-family: "JetBrains Mono", monospace;
    font-size: 9px;
    color: #A0A0A8;
    letter-spacing: 2.5px;
    text-transform: uppercase;
    margin: 8px 0 0;
}
</style>
""", unsafe_allow_html=True)

@st.cache_resource
def load_pipeline():
    corpus = pd.read_parquet(f"{PROCESSED}/corpus.parquet")
    corpus_ids = corpus["cord_uid"].tolist()
    id_to_title    = dict(zip(corpus["cord_uid"], corpus["title_clean"].fillna("")))
    id_to_abstract = dict(zip(corpus["cord_uid"], corpus["abstract_clean"].fillna("")))
    id_to_journal  = dict(zip(corpus["cord_uid"], corpus["journal"].fillna("Unknown")))
    id_to_year     = dict(zip(corpus["cord_uid"], corpus["publish_time"].fillna("")))
    id_to_cred     = dict(zip(corpus["cord_uid"], corpus["credibility"]))

    def tokenize(text):
        if not isinstance(text, str): return []
        return re.findall(r"\\b\\w+\\b", text.lower())
    bm25 = BM25Okapi([tokenize(t) for t in corpus["full_text"].tolist()])

    faiss_index = faiss.read_index(f"{EMB_PATH}/corpus_mE5.faiss")
    corpus_ids_npy = np.load(f"{EMB_PATH}/corpus_ids.npy", allow_pickle=True).tolist()
    mE5 = SentenceTransformer("intfloat/multilingual-e5-large", device="cuda")
    sentence_scorer = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2",
                                    max_length=512, device="cuda")

    return {
        "corpus_ids": corpus_ids, "corpus_ids_npy": corpus_ids_npy,
        "id_to_title": id_to_title, "id_to_abstract": id_to_abstract,
        "id_to_journal": id_to_journal, "id_to_year": id_to_year,
        "id_to_cred": id_to_cred,
        "bm25": bm25, "tokenize": tokenize,
        "faiss_index": faiss_index, "mE5": mE5,
        "sentence_scorer": sentence_scorer,
    }

def clean_snippet(text, max_len=200):
    """Strip HTML/XML tags and clean tracking strings from abstract sentences."""
    if not text:
        return ""
    text = re.sub(r"<[^>]+>", "", text)
    text = re.sub(r"\\(PsycInfo Database Record[^)]*\\)", "", text)
    text = re.sub(r"\\(c\\)\\s*\\d{4}[^.]*", "", text)
    text = re.sub(r"all rights reserved\\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"Impact Statement\\s*[:\\-]?\\s*", "", text)
    text = re.sub(r"Public Significance Statement[\\-:]?\\s*", "", text)
    text = re.sub(r"\\s+", " ", text).strip()
    if len(text) > max_len:
        text = text[:max_len].rsplit(" ", 1)[0] + "..."
    return text

def hybrid_search(tweet, P, top_k=5, fuse_pool=100, k_rrf=60):
    q = re.sub(r"http\\S+|www\\S+", "", tweet)
    q = re.sub(r"@\\w+", "", q)
    q = re.sub(r"\\s+", " ", q).strip()

    q_emb = P["mE5"].encode([f"query: {q}"], normalize_embeddings=True,
                              convert_to_numpy=True).astype("float32")
    dense_scores, dense_idx = P["faiss_index"].search(q_emb, fuse_pool)
    top1_dense_score = float(dense_scores[0][0])

    bm25_scores = P["bm25"].get_scores(P["tokenize"](q))
    bm25_top = np.argsort(bm25_scores)[-fuse_pool:][::-1]

    bm25_ranks  = {P["corpus_ids"][idx]: rank for rank, idx in enumerate(bm25_top, start=1)}
    dense_ranks = {P["corpus_ids_npy"][idx]: rank for rank, idx in enumerate(dense_idx[0], start=1)}

    fused = {}
    for cid in set(bm25_ranks) | set(dense_ranks):
        s = 0.0
        if cid in bm25_ranks:  s += 1.0 / (k_rrf + bm25_ranks[cid])
        if cid in dense_ranks: s += 1.0 / (k_rrf + dense_ranks[cid])
        fused[cid] = s

    results = sorted(fused.items(), key=lambda x: x[1], reverse=True)[:top_k]
    return results, top1_dense_score

def get_justification(tweet, cord_uid, P):
    abstract = P["id_to_abstract"].get(cord_uid, "")
    if not abstract or len(abstract) < 20:
        return clean_snippet(P["id_to_title"].get(cord_uid, "")), 0.0
    sentences = [s for s in sent_tokenize(abstract) if len(s) > 30]
    if not sentences:
        return clean_snippet(P["id_to_title"].get(cord_uid, "")), 0.0
    pairs = [[tweet, s[:500]] for s in sentences]
    scores = P["sentence_scorer"].predict(pairs)
    best = int(np.argmax(scores))
    return clean_snippet(sentences[best]), float(scores[best])

# Header
st.markdown(\'<p class="ct-tag">Scientific claim retrieval</p>\', unsafe_allow_html=True)
st.markdown(\'<div class="ct-title-wrap"><h1 class="ct-title">CheckThat<span class="ct-title-accent">!</span></h1></div>\', unsafe_allow_html=True)
st.markdown(\'<p class="ct-sub">Find the scientific paper behind any health claim.</p>\', unsafe_allow_html=True)

with st.spinner("Loading retrieval pipeline..."):
    P = load_pipeline()

examples = {
    "English": "wearing face masks reduces respiratory virus transmission",
    "اردو":    "چہرے کے ماسک سانس کی وائرل بیماری کی منتقلی کو کم کرتے ہیں",
    "العربية": "ارتداء الكمامات يقلل من انتقال الفيروسات التنفسية",
    "हिन्दी":   "फेस मास्क पहनने से सांस के वायरस का संचरण कम होता है",
}

cols = st.columns(len(examples))
for i, (lang, ex_text) in enumerate(examples.items()):
    if cols[i].button(lang, key=f"ex_{i}", use_container_width=True):
        st.session_state["tweet_input"] = ex_text

tweet = st.text_area(
    "Enter a tweet or claim",
    value=st.session_state.get("tweet_input", ""),
    height=100,
    key="tweet_input",
    placeholder="Paste a tweet in any language..."
)

col_a, col_b, col_c = st.columns([2, 1, 2])
with col_b:
    search_clicked = st.button("Search", type="primary", use_container_width=True)

if search_clicked and tweet.strip():
    with st.spinner("Searching corpus and extracting justifications..."):
        results, top1_score = hybrid_search(tweet, P, top_k=5)

    if top1_score < CONFIDENCE_THRESHOLD:
        st.markdown(f"""
        <div class="ct-noresults">
            <div class="ct-noresults-icon">🔍</div>
            <h3 class="ct-noresults-title">No relevant scientific papers found</h3>
            <p class="ct-noresults-body">
                Your query does not appear to match any paper in our COVID-19 / biomedical corpus.
                Try a claim about health, medicine, infectious disease, or pandemic response.
            </p>
            <p class="ct-noresults-meta">
                confidence {top1_score:.3f} · below threshold {CONFIDENCE_THRESHOLD}
            </p>
        </div>
        """, unsafe_allow_html=True)
    else:
        confidence_label = "high" if top1_score >= 0.88 else "medium"
        results_header_html = f"""
        <div class="ct-results-header">
            <span>5 results · 7,718 papers</span>
            <span>confidence <strong>{confidence_label}</strong> ({top1_score:.2f})</span>
        </div>
        """
        st.markdown(results_header_html, unsafe_allow_html=True)

        for rank, (cid, fusion_score) in enumerate(results, start=1):
            title    = P["id_to_title"].get(cid, "")
            journal  = P["id_to_journal"].get(cid, "Unknown")
            year     = str(P["id_to_year"].get(cid, ""))[:4]
            cred     = P["id_to_cred"].get(cid, 0.5)
            snippet, snip_score = get_justification(tweet, cid, P)

            display_title = title if len(title) <= 110 else title[:107] + "..."

            if cred >= 0.75:   dot_color = "#0F6E56"
            elif cred >= 0.55: dot_color = "#EF9F27"
            else:              dot_color = "#D85A30"

            card_html = f"""
            <div class="ct-card">
                <p class="ct-card-num">N° {rank:02d} — {journal[:30]} — {year}</p>
                <p class="ct-card-title">{display_title}</p>
                <p class="ct-card-snippet">"{snippet}"</p>
                <div class="ct-card-meta">
                    <span class="ct-card-meta-item">
                        <span class="ct-dot" style="background:{dot_color}"></span>
                        credibility · {cred:.2f}
                    </span>
                    <span class="ct-card-meta-item">match · {snip_score:.2f}</span>
                    <span class="ct-card-meta-item">id · {cid}</span>
                </div>
            </div>
            """
            st.markdown(card_html, unsafe_allow_html=True)

elif search_clicked:
    st.warning("Please enter a tweet first.")

# Footer
st.markdown(\'\'\'
<div class="ct-footer">
    <h3 class="ct-footer-name">CheckThat<span class="ct-footer-name-accent">!</span></h3>
    <div class="ct-footer-divider">
        <div class="ct-footer-line"></div>
        <div class="ct-footer-dot"></div>
        <div class="ct-footer-line"></div>
    </div>
    <p class="ct-footer-tag">For every claim, a source.</p>
    <p class="ct-footer-credit">Air University Islamabad &nbsp;·&nbsp; CLEF 2025</p>
</div>
\'\'\', unsafe_allow_html=True)
'''

with open('/content/app.py', 'w') as f:
    f.write(app_code)

print("✓ app.py FINAL — all features integrated")
print(f"  File size: {len(app_code)} bytes")

✓ app.py FINAL — all features integrated
  File size: 15602 bytes


In [ ]:
# === UI Cell 3: Launch Streamlit + ngrok tunnel ===
# Get your free ngrok auth token from: https://dashboard.ngrok.com/get-started/your-authtoken

import os, time, subprocess
from pyngrok import ngrok, conf

# 1. Set your ngrok auth token (get it free from https://ngrok.com)
NGROK_AUTHTOKEN = "YOUR_NGROK_TOKEN_HERE"  # <-- PASTE YOUR TOKEN HERE
conf.get_default().auth_token = NGROK_AUTHTOKEN

# 2. Kill any old Streamlit processes
subprocess.run(["pkill", "-f", "streamlit"], stderr=subprocess.DEVNULL)
time.sleep(2)

# 3. Kill any old ngrok tunnels
try:
    ngrok.kill()
    time.sleep(1)
except Exception:
    pass

# 4. Launch Streamlit in the background
subprocess.Popen(
    ["streamlit", "run", "/content/app.py",
     "--server.port", "8501",
     "--server.headless", "true",
     "--browser.gatherUsageStats", "false"],
    stdout=open("/content/streamlit.log", "w"),
    stderr=subprocess.STDOUT,
)
time.sleep(6)

# 5. Open ngrok tunnel to port 8501
print("Opening ngrok tunnel...")
public_url = ngrok.connect(8501)
print(f"\nApp is live at: {public_url}")
print("Note: URL changes every time you restart. Free tier only.")
